# 02 — Extraction des tableaux, textes et passages

**Objectif :** produire les données propres qui seront indexées.

Ce notebook montre pourquoi `table.extract()` ne suffit pas, puis utilise le
module `maintidoc.ingest` pour générer :

- `document.json` : textes, tableaux, images ;
- `tables.json` : tableaux nettoyés ;
- `chunks.json` : passages texte/tableaux avec métadonnées.


## Étape 1 — Comparer `extract()` et `get_textbox()`

`table.extract()` est rapide mais peut coller les mots. `page.get_textbox(rect)`
lit le texte dans le rectangle exact de la cellule et conserve mieux les espaces.

On compare les deux sur le premier tableau de la page 2.


In [1]:
import pymupdf
from maintidoc.config import PDF_PATH

with pymupdf.open(PDF_PATH) as doc:
    page = doc[1]
    table = page.find_tables().tables[0]

    print("=== extract() ===")
    for row in table.extract()[:3]:
        print(row)

    print("\n=== get_textbox() ===")
    for row in table.rows[:3]:
        cells = []
        for cell in row.cells:
            if cell is None:
                cells.append("")
            else:
                cells.append(" ".join(page.get_textbox(pymupdf.Rect(cell)).split()))
        print(cells)

=== extract() ===
['Champ', 'Valeurpedagogique']
['Equipement', 'SAN -LAB 2000 (fictif)']
['Controleurs', 'AetB hautedisponibiliteconceptuelle\n,']

=== get_textbox() ===
['Champ', 'Valeur pedagogique']
['Equipement', 'SAN-LAB 2000 (fictif)']
['Controleurs', 'A et B, haute disponibilite conceptuelle']


### Lecture du résultat

`extract()` produit `Valeurpedagogique`, `AetB`, etc. `get_textbox()` produit
`Valeur pedagogique`, `A et B`, etc.

Conclusion : on garde `get_textbox()` pour construire les tableaux.


## Étape 2 — Lancer l'ingestion complète

`extract_pdf()` parcourt le PDF, construit les enregistrements de texte, les
tableaux nettoyés et les images. `build_chunks()` découpe ensuite les textes en
passages avec chevauchement.

Le découpage est nécessaire : un document entier est trop long pour être comparé
efficacement à une question.


In [2]:
from maintidoc.ingest import build_chunks, extract_pdf, save_artifacts

document = extract_pdf()
chunks = build_chunks(document)
save_artifacts(document, chunks)

print("Pages :", len(document["texts"]))
print("Tableaux :", len(document["tables"]))
print("Images :", len(document["images"]))
print("Chunks :", len(chunks))

Pages : 9
Tableaux : 9
Images : 6
Chunks : 26


### Lecture du résultat

On attend 9 pages, 9 tableaux, 6 images et environ 26 passages. Le nombre exact de
chunks dépend de la taille des pages.


## Étape 3 — Vérifier les tableaux

On affiche le premier tableau et sa représentation Markdown. La représentation
Markdown est celle qui sera envoyée au modèle et indexée.


In [3]:
for table in document["tables"][:2]:
    print("=" * 70)
    print(table["element_id"], "page", table["page"])
    print(table["text"][:900])

P02-TAB01 page 2
| Champ | Valeur pedagogique |
|---|---|
| Equipement | SAN-LAB 2000 (fictif) |
| Controleurs | A et B, haute disponibilite conceptuelle |
| Connectivite | 2 fabrics FC A/B; zoning et LUN masking |
| Protection donnees | Groupe de disques protege; details RAID non specifies |
| Supervision | Etat controleurs, disques, ports, chemins et capacite |
| Exclusions | Aucune instruction de remplacement a chaud ni modification de zoning en production |
P03-TAB01 page 3
| Terme | Definition / verification |
|---|---|
| HBA | Carte de connexion FC cote hote; etat du port et WWPN |
| WWPN | Identifiant de port Fibre Channel; correspondance inventaire |
| Zoning | Regles de visibilite des ports dans une fabric |
| LUN masking | Autorisation d’acces aux volumes par les initiateurs |
| MPIO | Gestion des chemins multiples et basculement selon configuration |


### Lecture du résultat

Les cellules doivent être lisibles : `SAN-LAB 2000`, `A et B`, `haute disponibilité`.
Les identifiants de tableau (`P02-TAB01`) et les pages sont conservés pour les
citations.


## Étape 4 — Vérifier les passages

Chaque chunk contient :

- `chunk_id` : identifiant interne ;
- `source_element_id` : identifiant citable (`P02-TEXT`) ;
- `page` ;
- `type` : texte ou tableau ;
- `text`.


In [4]:
import pandas as pd

df_chunks = pd.DataFrame(chunks)
print(df_chunks[["chunk_id", "source_element_id", "page", "type"]].head(10))
print("\nLongueur moyenne :", round(df_chunks["text"].str.len().mean(), 1))
print("Types :", df_chunks["type"].value_counts().to_dict())

       chunk_id source_element_id  page  type
0  P01-TEXT-C01          P01-TEXT     1  text
1  P02-TEXT-C01          P02-TEXT     2  text
2  P02-TEXT-C02          P02-TEXT     2  text
3  P03-TEXT-C01          P03-TEXT     3  text
4  P03-TEXT-C02          P03-TEXT     3  text
5  P04-TEXT-C01          P04-TEXT     4  text
6  P04-TEXT-C02          P04-TEXT     4  text
7  P05-TEXT-C01          P05-TEXT     5  text
8  P05-TEXT-C02          P05-TEXT     5  text
9  P06-TEXT-C01          P06-TEXT     6  text

Longueur moyenne : 500.6
Types : {'text': 17, 'table': 9}


### Lecture du résultat

Les passages doivent être courts (quelques centaines de caractères), avec une page
et un identifiant source. Les tableaux sont des chunks séparés.


## Étape 5 — Architecture des artefacts

```text
data/processed/
├── document.json       texte + tableaux + images
├── tables.json         tableaux nettoyés
├── texts.json          texte brut par page
├── chunks.json         passages indexables
├── inventory.csv       inventaire des pages
└── images/             couverture + FIG-01..FIG-05
```

Ces fichiers sont la source unique de l'indexation. Les notebooks suivants ne
relisent pas le PDF : ils lisent ces artefacts.


## Bilan

- Les tableaux sont reconstruits avec `get_textbox()`.
- Les 9 pages, 9 tableaux et 6 images sont conservés.
- 26 passages sont prêts pour BGE-M3.
- Les identifiants de citation sont stables.

**Étape suivante :** indexer les passages et les schémas, puis fusionner les
résultats.
